# Tensors: a quickstart

`gb.Tensor` is an n-dimensional sparse array: a Matrix and a Vector generalized to any number of axes. It follows GraphBLAS for missing values (there is no fill value) and numpy for shapes, axes and broadcasting.

**The Tensor is experimental**: names, defaults and behavior may change in any release. It needs the `suitesparse` backend with SuiteSparse:GraphBLAS 10 or later. The user guide's "Tensors" page is the reference; this notebook walks through it on one small example.

In [1]:
import numpy as np

import graphblas as gb
from graphblas import Tensor, binary, einsum, semiring

## The model: a logical shape, stored in one Matrix

Take messages between four people over a day: `M[sender, receiver, hour]` is how many messages `sender` sent `receiver` in that hour. Most of the 4 x 4 x 24 positions are empty.

In [2]:
senders = [0, 0, 1, 2, 2, 3, 3, 3, 1]
receivers = [1, 2, 0, 3, 1, 0, 1, 2, 3]
hours = [9, 9, 10, 14, 15, 9, 21, 21, 9]
counts = [2, 1, 4, 1, 3, 2, 1, 5, 2]
M = Tensor.from_coo([senders, receivers, hours], counts, shape=(4, 4, 24), name="M")
M

gb.Tensor  "M"  nvals 9  shape (4, 4, 24)  dtype INT64  layout (0|1,2)  format csr
entries:
  (0, 1, 9): 2
  (0, 2, 9): 1
  (1, 0, 10): 4
  (1, 3, 9): 2
  (2, 1, 15): 3
  (2, 3, 14): 1
  (3, 0, 9): 2
  (3, 1, 21): 1
  (3, 2, 21): 5

A Tensor stores its entries in one SuiteSparse Matrix, held by row, plus a **layout**: which axes are raveled (in C order) into the Matrix's rows and which into its columns. `(0|1,2)` means the sender indexes the rows, and receiver and hour together index the columns. The shape is always the logical one; the layout is only how it is stored.

In [3]:
M.layout, M.ss.matrix.shape

(Layout(rows=(0,), cols=(1, 2)), (4, 96))

Operations move the data to another layout when they need one (a planned "relayout"), so the layout is mostly something to know about, not to manage. `relayout` chooses one explicitly; the Tensor is the same.

In [4]:
by_hour_rows = M.relayout(((2,), (0, 1)))
by_hour_rows.layout, by_hour_rows.ss.matrix.shape, by_hour_rows.isequal(M)

(Layout(rows=(2,), cols=(0, 1)), (24, 16), True)

## Construction

Besides `from_coo`, a Tensor comes from a dense array (with the value that means "missing"), from Matrices stacked along a new axis, or empty. `to_coo`, `to_dense`, `to_matrix`, `to_matrices` and friends convert back.

In [5]:
D = Tensor.from_dense(np.arange(6.0).reshape(1, 2, 3), missing_value=0.0)
D.nvals, D.to_dense(fill_value=0)

(5,
 array([[[0., 1., 2.],
         [3., 4., 5.]]]))

In [6]:
nine_am = gb.Matrix.from_coo([0, 0, 1, 3], [1, 2, 3, 0], [2, 1, 2, 2], nrows=4, ncols=4)
ten_am = gb.Matrix.from_coo([1], [0], [4], nrows=4, ncols=4)
morning = Tensor.from_matrices([nine_am, ten_am], axis=2)  # (sender, receiver, hour 9 or 10)
morning.shape, morning.isequal(M[:, :, 9:11].new())

((4, 4, 2), True)

In [7]:
E = Tensor(gb.dtypes.FP64, (4, 4, 24))  # empty
E.nvals

0

## Views, reshape and copy-on-write

Shape operations change only the shape and layout, in O(1), and share the stored data. Tensors that share data copy it before one of them changes, so a Tensor never sees another's writes.

In [8]:
H = M.permute_dims((2, 0, 1))  # (hour, sender, receiver)
H.shape, H.layout

((24, 4, 4), Layout(rows=(1,), cols=(2, 0)))

In [9]:
H[9, 0, 1] = 100  # writing to the view copies first
H[9, 0, 1].new().to_scalar().value, M[0, 1, 9].new().to_scalar().value

(100, 2)

`reshape` (C order) is a view when each run of axes it splits or fuses lies within one group of the layout. Splitting the hours into four periods of six, or fusing receiver and hour (both in the columns), are views. Fusing sender (rows) with receiver (columns) moves the data, and `copy=False` refuses that, as in the Array API.

In [10]:
M.reshape(4, 4, 4, 6).shape, M.reshape(4, 96).layout

((4, 4, 4, 6), Layout(rows=(0,), cols=(1,)))

In [11]:
try:
    M.reshape(16, 24, copy=False)
except ValueError as exc:
    print(exc)
M.reshape(16, 24).shape  # without copy=False, the data moves once

reshape from (4, 4, 24) to (16, 24) with copy=False: the layout (0|1,2) does not hold the result, so the data would move


(16, 24)

## Element-wise operations and broadcasting

Element-wise operations follow the Matrix and Vector ones: `ewise_mult` (and `*`, `&`) on the intersection of entries, `ewise_add` (and `+`, `|`) on the union, `ewise_union` on the union with a default for a missing side. Operands broadcast as in numpy, and a Matrix, Vector or ndarray operand follows the Tensor's rules. Expressions are lazy: `.new()` computes them.

In [12]:
night = gb.Vector.from_coo(list(range(6)) + list(range(20, 24)), 2.0, size=24)
night_weighted = M.ewise_add(M * night, binary.max).new()  # night messages count twice
night_weighted[3, :, 21].new().to_dense(fill_value=0)

array([ 0.,  2., 10.,  0.])

In [13]:
M.apply(binary.times, 10).new().to_coo()[1], M.select(gb.select.valuegt, 2).new().nvals

(array([20, 10, 40, 20, 30, 10, 20, 10, 50]), 3)

## Reductions

`reduce` takes a monoid (or an aggregator that is one) and any set of axes. A fiber with no entries stays missing, as in GraphBLAS.

In [14]:
sent = M.sum(axis=(1, 2)).new()  # per sender
received = M.sum(axis=(0, 2)).new()  # per receiver
sent.to_dense(fill_value=0), received.to_dense(fill_value=0)

(array([3, 6, 4, 8]), array([6, 6, 6, 3]))

In [15]:
per_hour = M.sum(axis=(0, 1)).new()
per_hour.to_coo()

((array([ 9, 10, 14, 15, 21]),), array([7, 4, 1, 3, 6]))

In [16]:
active_hours = M.reduce(gb.agg.count, axis=2).new()  # hours in which each pair talked
active_hours.to_dense(fill_value=0)

array([[0, 1, 1, 0],
       [1, 0, 0, 1],
       [0, 1, 0, 1],
       [1, 1, 1, 0]])

## Contractions: tensordot, einsum, matmul and MTTKRP

`tensordot` contracts axes with a semiring (`plus_times` by default). Here a 24 x 4 indicator matrix sums the hours into four six-hour periods: a TTM (tensor times matrix).

In [17]:
P = np.zeros((24, 4))
P[np.arange(24), np.arange(24) // 6] = 1  # hour -> period
per_period = M.tensordot(P, ([2], [0])).new()  # (sender, receiver, period)
per_period[:, :, 1].new().to_dense(fill_value=0)  # 6:00 to 12:00

array([[0., 2., 1., 0.],
       [4., 0., 0., 2.],
       [0., 0., 0., 0.],
       [2., 0., 0., 0.]])

`einsum` takes numpy's subscripts and any semiring. The messages each person sent in each period, and, with the `max_times` semiring, the most they sent anyone in one hour of it:

In [18]:
einsum("srh,hp->sp", M, P).new().to_dense(fill_value=0)

array([[0., 3., 0., 0.],
       [0., 6., 0., 0.],
       [0., 0., 4., 0.],
       [0., 2., 0., 6.]])

In [19]:
einsum("srh,hp->sp", M, P, op=semiring.max_times).new().to_dense(fill_value=0)

array([[0., 2., 0., 0.],
       [0., 4., 0., 0.],
       [0., 0., 3., 0.],
       [0., 2., 0., 5.]])

`@` contracts the last axis of the left operand with the second-to-last of the right one, with the leading axes as batch axes. With the hour first, `H @ H` counts, for each hour, the two-message relays `s -> m -> r` within that hour (weighted by the counts):

In [20]:
H = M.permute_dims((2, 0, 1))
relays = (H @ H).new()
relays[9].new().to_dense(fill_value=0)

array([[0, 0, 0, 4],
       [4, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 4, 2, 0]])

`mttkrp`, the core of CP decompositions, never forms the Khatri-Rao product. It runs a Numba kernel when Numba is installed, else GraphBLAS products; the result is the same.

In [21]:
rng = np.random.default_rng(0)
factors = [rng.random((n, 2)) for n in M.shape]
Mf = M.dup(float)  # the kernel takes float64 values
Mf.mttkrp(factors, 0).new().to_dense()

array([[1.30766543, 0.01211885],
       [1.82310426, 3.43593364],
       [0.84172893, 0.16150149],
       [1.74812983, 0.58907553]])

## Masks, accumulation and assignment

Updates work as for Matrix: `C(mask, accum, replace) << expression`. A mask is a Tensor's `.S` (structure) or `.V` (values), or a Matrix's or Vector's of the same shape.

In [22]:
C = Tensor(gb.dtypes.INT64, M.shape)
C(M.S) << M.apply(binary.times, 10)  # only where M has an entry
C(accum=binary.plus) << M
C.to_coo()[1]

array([22, 11, 44, 22, 33, 11, 22, 11, 55])

`T[...] = value` and `T[...] << value` assign to the region an outer index selects, with GraphBLAS assign semantics. The value broadcasts against the region as in numpy. `T(mask)[...] << value` is an assign (the mask covers all of `T`); `T[...](mask) << value` is a subassign (the mask covers the region).

In [23]:
Y = Tensor(gb.dtypes.INT64, (4, 4, 24))
Y[0, :, 9:12] = 1  # sender 0, every receiver, 9:00 to 12:00
Y(accum=binary.plus)[0, 1, :] << 5  # add 5 to every hour for the pair (0, 1)
Y[0, 1].new().to_dense(fill_value=0)

array([5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5])

In [24]:
del Y[0, 1, 12:]  # delete a region
Y[0](Y[0].new().S) << 7  # a subassign: only where sender 0 already has entries
Y[0].new().to_dense(fill_value=0)[:, 8:13]

array([[0, 7, 7, 7, 0],
       [7, 7, 7, 7, 0],
       [0, 7, 7, 7, 0],
       [0, 7, 7, 7, 0]])

## Indexing and gathering

`T[...]` extracts with outer (orthogonal) indexing, as GraphBLAS extract does: ints drop their axis, while slices, lists and integer arrays keep it.

In [25]:
M[3].new().shape, M[:, [0, 1], 9:12].new().shape, M[0, 1, 9].new().to_scalar().value

((4, 24), (4, 2, 3), 2)

`take`, `take_along_axis` and `vindex` gather with numpy's rules for integer indexing instead. A position that lands on a missing entry is missing in the result.

In [26]:
points = M.vindex[[0, 3, 2], [1, 2, 0], [9, 21, 9]]  # three (sender, receiver, hour) points
points.to_coo()

((array([0, 1]),), array([2, 5]))

In [27]:
M.take([9, 21], axis=2).shape

(4, 4, 2)

## Input and output

A Tensor converts to and from scipy's n-d `coo_array`, and pydata/sparse's COO and GCXS arrays (GCXS is the Tensor's own storage, its compressed axes as rows).

In [28]:
S = M.to_scipy_sparse()
type(S).__name__, S.shape, Tensor.from_scipy_sparse(S).isequal(M)

('coo_array', (4, 4, 24), True)

A 2-d Tensor is a Matrix: `to_matrix` and `from_matrix` convert, and a Matrix or Vector compares with a Tensor directly.

In [29]:
pair_totals = M.sum(axis=2).new().to_matrix()
pair_totals, pair_totals.isequal(M.sum(axis=2).new())

("M_93"     nvals  nrows  ncols  dtype   format
 gb.Matrix      9      4      4  INT64  bitmapr
 ----------------------------------------------
    0  1  2  3
 0     2  1   
 1  4        2
 2     3     1
 3  2  1  5   ,
 True)

The notebook "Tensor Multiplex PageRank" runs a real example: it ranks European airports over 37 airline networks at once.